# 01 — Data exploration

Inspect the aligned IMERG (10 km), AORC (1 km) and NLCD/DEM products produced by `python main.py preprocess`.

In [ ]:
import sys, numpy as np, pandas as pd, xarray as xr, matplotlib.pyplot as plt
sys.path.insert(0, '..')
from src.utils import load_config, grids_from_config, coarsen_mean, upsample_nearest
from src.validation import SEQ_CMAP, DIV_CMAP
cfg = load_config('../config.yaml'); grids = grids_from_config(cfg)
proc = cfg['paths']['processed']
imerg = xr.open_dataset(f'{proc}/imerg_aligned_10km.nc')['precip']
aorc = xr.open_dataset(f'{proc}/aorc_aligned_1km.nc', chunks={'time': 64})['precip']
aorc_c = xr.open_dataset(f'{proc}/aorc_aligned_10km.nc')['precip']
nlcd = xr.open_dataset(f'{proc}/nlcd_aligned_1km.nc')
print(grids.coarse.shape, grids.fine.shape, imerg.sizes)

## Static predictors

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
nlcd['dem'].plot(ax=axes[0], cmap='terrain'); axes[0].set_title('Elevation (m)')
nlcd['slope'].plot(ax=axes[1], cmap=SEQ_CMAP, vmax=float(nlcd['slope'].quantile(0.99))); axes[1].set_title('Slope (deg)')
nlcd['lc_frac_developed'].plot(ax=axes[2], cmap=SEQ_CMAP); axes[2].set_title('Developed fraction')
nlcd['imperv'].plot(ax=axes[3], cmap=SEQ_CMAP); axes[3].set_title('Impervious fraction')
plt.tight_layout()

## Climatology: IMERG vs AORC

In [ ]:
mean_imerg = imerg.mean('time')
mean_aorc_c = aorc_c.mean('time')
mean_aorc = aorc.mean('time').compute()
vmax = float(max(mean_imerg.max(), mean_aorc.quantile(0.99)))
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
mean_imerg.plot(ax=axes[0], cmap=SEQ_CMAP, vmin=0, vmax=vmax); axes[0].set_title('IMERG mean (mm/day)')
mean_aorc_c.plot(ax=axes[1], cmap=SEQ_CMAP, vmin=0, vmax=vmax); axes[1].set_title('AORC 10 km mean (mm/day)')
mean_aorc.plot(ax=axes[2], cmap=SEQ_CMAP, vmin=0, vmax=vmax); axes[2].set_title('AORC 1 km mean (mm/day)')
plt.tight_layout()
(mean_imerg - mean_aorc_c).plot(cmap=DIV_CMAP, vmin=-1.5, vmax=1.5, figsize=(6, 4)); plt.title('IMERG - AORC mean bias (mm/day, 10 km)');

## Domain-mean daily series and intensity distributions

In [ ]:
dm = pd.DataFrame({'imerg': imerg.mean(('lat', 'lon')).values, 'aorc_10km': aorc_c.mean(('lat', 'lon')).values}, index=imerg.time.values)
ax = dm.resample('MS').sum().plot(figsize=(13, 3.5), lw=1.5, color=['#1baf7a', '#0b0b0b']); ax.set_ylabel('mm/month')
print('Daily domain-mean correlation:', dm.corr().iloc[0, 1].round(3))
q = np.array([0.5, 0.9, 0.99, 0.999])
print(pd.DataFrame({'imerg': np.nanquantile(imerg.values, q), 'aorc_10km': np.nanquantile(aorc_c.values, q)}, index=q))

## Missing data

In [ ]:
print('IMERG missing fraction:', float(imerg.isnull().mean()))
print('Days with any IMERG missing:', int(imerg.isnull().any(('lat', 'lon')).sum()))
print('AORC 10 km missing fraction:', float(aorc_c.isnull().mean()))